# Lab 3 KNN 與決策樹：兩台分類器，同一把誠實的尺

**今天的目標：** 用上一個 Lab 整理好的那種財務特徵，親手做出**兩台**「猜漲跌」的分類器，然後用同一份資料、同一個切法比一比。你會走過四段：

- **A** 兩種判法各手刻一次：**找最像的鄰居** vs **問一連串問題**
- **B** 換上真股票資料，把兩台都寫成業界版本（你會發現寫法幾乎一樣）
- **C** ⭐ 課程重點：把兩台各自的旋鈕轉到極端，**看它們用同一種方式壞掉**
- **D** 收尾

> 🚫 **免責：** 今天做的兩台「猜漲跌」分類器，純為教學示範機器學習怎麼運作，**不是投資建議**。B 段用的雖然是真股價，但「猜隔日漲跌」樣本外只有 5 成、跟擲銅板差不多。今天學的是「模型怎麼運作 + 怎麼誠實驗證它」，不是「跟著模型買股票」。
> 🌱 **今天全班的數字應該一模一樣**——Part A 是手編資料，B 段之後大家讀的是同一份 `data/prices_cache.csv`。**跟老師不一樣**多半是快取被刪、重新上網抓了新資料，或決策樹漏了 `random_state=42`。

In [ ]:
# 📦 先跑這一格：一次裝齊今天要用的套件（已裝好的會直接跳過；裝不起來看 README）
!pip install -q scikit-learn pandas numpy yfinance

## 🔧 第 0 步：環境檢查

**預期輸出：** 印出三個套件的版本號——有版本號就代表裝好了。

> ⚠️ 套件名字叫 `scikit-learn`，但程式裡 `import` 的名字是 `sklearn`。

In [ ]:
import sklearn
import pandas as pd
import numpy as np

print("sklearn", sklearn.__version__)
print("pandas ", pd.__version__)
print("numpy  ", np.__version__)

---
## A・認識兩台分類器（直接用 sklearn）

老師剛剛講了咖啡廳的故事：**小美**翻出「跟他最像的 3 個人」來猜，**阿凱**則是「問兩題就八九不離十」。

這兩套做法，機器學習裡都有現成的：
- 小美「找最像的鄰居投票」＝ **KNN**（K 個最近鄰居）
- 阿凱「問一連串問題分岔」＝ **決策樹**

**這一段直接用 `sklearn` 把兩台都跑出來**——它們的內部原理老師在投影片講，你在這裡的任務是**看懂怎麼呼叫、看懂輸出在說什麼**。

### A0・sklearn 的鐵則：所有模型都長一樣

sklearn 把每個模型都做成**同一種介面**，所以你只要學一次，兩台都會用：

| 動作 | 意思 |
|---|---|
| `模型 = 某某Classifier(...)` | 建一台空模型，順便設定旋鈕 |
| `模型.fit(X, y)` | 拿「特徵 + 答案」餵進去學 |
| `模型.predict(新資料)` | 給新資料，回傳它猜的答案 |
| `模型.score(X, y)` | 算準確率（猜對幾筆 ÷ 總筆數） |

> 🆕 **`X` 和 `y` 的慣例：** 大寫 `X` ＝特徵表（每列一家公司、每行一個指標）；小寫 `y` ＝答案。等一下 A1-2 你會親手把資料拆成這兩半。

### A1・今天全程用的 6 家虛構公司

上一個 Lab 你把髒資料洗成一張 pandas 表格。今天延續同樣的方式：**把這 6 家公司放進一張 `DataFrame`**，每一列一家、每一欄一個財務數字，最後一欄是「示意漲（1）／跌（0）」的答案。

先解釋這三個數字（給沒碰過金融的人）：
- **本益比 PE**＝股價 ÷ 每股盈餘。白話：「買這檔，要幾年的獲利才回本」。
- **毛利率 margin**＝賣東西扣掉直接成本後賺的比例，**反映本業賺不賺錢**。
- **負債比 debt**＝總負債 ÷ 總資產，**反映財務風險**，越高代表借越多。

**預期輸出：** 一張 6 列的表。**前 3 家（高毛利、低負債）label 都是 1，後 3 家（低毛利、高負債）都是 0**——這是刻意設計的乾淨資料，等一下你才看得出兩台模型怎麼判。

In [ ]:
import pandas as pd

# 6 家【虛構・教學用】公司（不是真實行情，數字是編的）
companies = pd.DataFrame({
    "PE":     [12, 15, 11, 35, 40, 38],   # 本益比
    "margin": [45, 40, 50, 15, 12, 18],   # 毛利率
    "debt":   [30, 35, 28, 70, 75, 65],   # 負債比
    "label":  [1,  1,  1,  0,  0,  0],    # 1 = 示意漲、0 = 示意跌
})

def label_name(label):
    """把 1 / 0 翻成看得懂的字（後面每一格都會用到）"""
    if label == 1:
        return "漲"
    else:
        return "跌"

print(companies)

### A1-2・把表格拆成「特徵表 X」和「答案 y」

模型要吃兩樣東西：**拿來判斷的線索**（三個財務數字）和**要它猜的答案**（漲／跌）。所以先把表格拆成兩半：
- `X_small`＝三欄財務數字，就是**特徵表**（習慣用大寫 `X`）。
- `y_small`＝label 那一欄，就是**答案**（習慣用小寫 `y`）。

**今天兩台 sklearn 模型全部都吃這一組 `X_small` / `y_small`。**

**預期輸出：** 上面是 6 筆各 3 個數字的清單，下面是 6 個 0/1。

#### 先體會一下：只挑三欄、還沒拆成 X/y 之前長什麼樣子

在拆成 X / y 之前，先只把**三欄財務數字**（不含 label）挑出來看一眼——它還是一張 pandas 表格。

**預期輸出：** 一張 6 列 × 3 欄（PE / margin / debt）的表，**沒有 label 那一欄**。

In [ ]:
feature_names = ["PE", "margin", "debt"]
# TODO：只挑這三欄（用 feature_names 這個清單去取），label 不要
#       最後一行不用 print，讓 Jupyter 直接把子表顯示出來
companies[____]

#### 再看 `.values` 和 `.tolist()` 的差別

模型不吃 pandas 表格，要餵**純數字的 list**。這裡分兩步看它怎麼變過去：
- `.values` → 把表格變成 **numpy 陣列**（印出來有 `array([...])` 的外框）。
- `.values.tolist()` → 再把陣列變成**一般的 Python 巢狀 list**（乾淨的 `[[...], [...]]`）。

**預期輸出：** 上面一坨有 `array(` 外框，下面一坨是乾淨的中括號清單。盯著看差別。

In [ ]:
# TODO：加一個屬性，把表格變成數字陣列（印出來會有 array 外框）
print(companies[feature_names].____)

In [ ]:
# TODO：先接上一格那個屬性（→ 數字陣列），再接一個方法把陣列變成一般 list（記得括號）
print(companies[feature_names].____.____())

In [ ]:
# 有了上面的體會，這一格直接組好今天要用的兩樣東西
X_small = companies[feature_names].values.tolist()   # 特徵表：每筆 3 個數字
y_small = companies["label"].tolist()                # 答案：每筆 1 個 0/1

print("X_small =", X_small)
print("y_small =", y_small)

> **🔀 小整理 ①：特徵 vs 答案（最常搞混）**
>
> | | 特徵（feature）＝`X` | 答案（label）＝`y` |
> |---|---|---|
> | 是什麼 | 描述樣本的數字（**輸入**） | 想預測的分類（**輸出**） |
> | 情境劇裡 | 住多遠 / 有沒有經驗 / 時薪 | 做滿 / 沒做滿 |
> | 今天的例子 | 本益比、毛利率、負債比 | 示意漲（1）/ 示意跌（0） |
>
> **一句話：** `X` 是「你給模型看的線索」，`y` 是「你要模型猜的答案」。

### A2・第一台：KNN（小美的做法）

`KNeighborsClassifier(n_neighbors=3)`＝「看最近 3 家鄰居投票」。`n_neighbors` 就是老師講的 `k`。

**預期輸出：** 這家體質像「漲」那群 → 判漲。

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# TODO：把「看最近幾家鄰居」的參數設成 3。參數名不是 k —— 回頭看 A0 表格 / A2 說明的寫法
knn = KNeighborsClassifier(____=3)
# TODO：三動作的第一個 —— 把資料餵進模型
knn.____(X_small, y_small)

#### 這一格的「預測資料」是怎麼設計的

下一格要預測的公司是 `[13, 43, 32]`——**低本益比、高毛利、低負債**，故意設計成很像 A1 那份資料裡「漲」那一群（前 3 家）。所以答案應該是「漲」，你可以拿它檢查自己填對沒。

還有 `predict([[13, 43, 32]])` 的**兩層中括號**：sklearn 規定「一次預測一批」，就算只猜 1 家，也要把它包成「裡面只有一家的清單」。回傳也是清單，所以取 `pred[0]`。

In [ ]:
# TODO：用哪個動作叫模型「猜」？（三動作之一；[[...]] 已經幫你包成一批了）
pred = knn.____([[13, 43, 32]])
print("新公司 [13,43,32] 預測：", label_name(pred[0]))

### A3・第二台：決策樹（阿凱的做法）

**盯著看：`fit`、`predict` 跟上一格 KNN 一字不差，只有第一行換了模型名字。** 這就是「同一個介面」的意思。

**預期輸出：** 體質好的判漲、體質差的判跌。

> ⚠️ 決策樹**建樹時有隨機性**，所以要多帶一個 `random_state=42`（KNN 不用，它沒有隨機性）。

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

# TODO：限制樹「最多問幾層問題」的參數，設成 2（就是 A0 講的決策樹旋鈕）
tree = DecisionTreeClassifier(____=2, random_state=42)
# TODO：餵資料 —— 跟 KNN 一模一樣的那個動作
tree.____(X_small, y_small)

In [ ]:
new = [13, 43, 32]                            # 體質好的公司（高毛利、低負債）
# TODO：跟剛剛 KNN 一模一樣的動作（同一個介面！）；new 一樣要包成一批
pred = tree.____([new])
print("新公司", new, "預測：", label_name(pred[0]))

In [ ]:
new = [37, 16, 68]                            # 體質差的公司（低毛利、高負債）
pred = tree.predict([new])                    # 這格直接給，體會「同一段寫法換一家公司」
print("新公司", new, "預測：", label_name(pred[0]))

### A4・決策樹專屬好料一：把樹印成文字規則

決策樹最大的賣點是**好解釋**——整棵樹可以攤成一串縮排的 if/else，你能直接讀給老闆聽。

**預期輸出：** 只有一層！`margin <= 29.00 → 跌`、`margin > 29.00 → 漲`。

> ⚠️ **盯著看：我們明明給它 `max_depth=2`（允許問 2 層），它卻只問了一題就停。** 為什麼？下一格的輸出會給你答案。

In [ ]:
print(export_text(tree, feature_names=feature_names))
# 💡 沒傳 feature_names 的話會印成 feature_0 / feature_1 / feature_2，看不出是哪個指標

### A5・決策樹專屬好料二：哪個指標最重要

決策樹訓練完會告訴你「**哪個指標對分類最有貢獻**」——被拿來分岔越多次、分得越乾淨的，數字越大；三個加起來 = 1.0。

**預期輸出：** `margin` 是 1.0，`PE` 和 `debt` 都是 0.0。

> ⚠️ **看到 0.0 不要以為壞掉了。** 它的意思是「**那個指標根本沒被拿來問**」。
> 為什麼樹只看毛利率？**因為這 6 家裡「沒有任何一家是高毛利卻下跌」** —— 資料沒給它「高負債會拖垮高毛利」的例子，它就學不到。**模型只能從看過的資料學。**

In [ ]:
for i in range(len(feature_names)):
    # TODO：樹訓練完會有一個屬性，裝著「每個指標的重要性」（結尾有底線）
    print(feature_names[i], "重要性：", round(tree.____[i], 3))
# 想一想：為什麼有的指標是 0.0？

> **🔀 小整理 ②：KNN vs 決策樹（小美 vs 阿凱）**
>
> | | KNN（小美：找最像的） | 決策樹（阿凱：問問題） |
> |---|---|---|
> | 怎麼判 | 看最近 k 個鄰居投票 | 問一連串 if/else 問題、走到葉 |
> | `fit` 在做 | 只「記住」資料（懶惰） | **真的算**：找問題、建整棵樹 |
> | 過擬合旋鈕 | `n_neighbors`（k） | `max_depth`（樹深度） |
> | 要固定 `random_state` 嗎 | 不用（沒隨機性） | **要**（建樹有隨機性） |
> | 額外好料 | （無） | `export_text` ＋ `feature_importances_` |
>
> **一句話：** `fit` / `predict` / `score` 三個動作完全一樣，所以你學一次就兩台都會用；差別只在「裡面是鄰居還是樹」。

### 📝 小作業 A

你已經有兩台訓練好的模型（`knn`、`tree`）。拿同一家公司分別餵給它們——**兩台會不會給出不同答案？** 試著找出一家讓它們吵架的公司。

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
for c in [[13, 43, 32], [37, 16, 68], [25, 30, 50]]:
    k = label_name(knn.predict([c])[0])
    t = label_name(tree.predict([c])[0])
    print("公司", c, "→ KNN 判", k, "、決策樹 判", t)
```
**結論：** 前兩家兩台一致；但 **`[25,30,50]` KNN 判跌、決策樹判漲——兩台吵架了。**
為什麼？決策樹只看「毛利率 > 29 就判漲」，這家 margin=30 剛好過線 → 判漲；KNN 看最近的鄰居，它們偏「跌」那群 → 判跌。**同一家公司、兩台不同答案，那到底信誰？** 這正是 C 段「怎麼誠實比較兩台」要回答的事。
</details>

In [ ]:
# 📝 小作業 A：你的答案（拿同一家公司餵兩台，找出「兩台不同意」的那家）

for c in [[13, 43, 32], [37, 16, 68], [25, 30, 50]]:
    # TODO：分別叫 knn 和 tree 去「猜」這家公司 c（跟前面一樣要包成一批、取 [0]）
    k = label_name(knn.____([c])[0])
    t = label_name(tree.____([c])[0])
    print("公司", c, "→ KNN 判", k, "、決策樹 判", t)
# 找找看：哪一家兩台給的答案不一樣？

### 📝 小作業 B（⭐ 進階選做）

把 A3 的 `max_depth` 從 `2` 改成 `1`，重新訓練一棵樹，再印一次規則。**印出來的規則會變嗎？為什麼？**

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
tree1 = DecisionTreeClassifier(max_depth=1, random_state=42)
tree1.fit(X_small, y_small)
print(export_text(tree1, feature_names=feature_names))
```
**結論：一模一樣。** 因為這 6 家用毛利率一刀就分乾淨了，**樹本來就只想問一題**——`max_depth=2` 只是「允許」問到 2 層，不是「規定」要問滿 2 層。

**這件事下一段會反過來咬你：** 當資料變亂、一刀分不乾淨的時候，你給它多深，它就會鑽多深。
</details>

In [ ]:
# 📝 小作業 B：你的答案（把樹的深度上限改成 1，再印一次規則，比比看有沒有變）

tree1 = DecisionTreeClassifier(____=1, random_state=42)
tree1.fit(X_small, y_small)
print(export_text(tree1, feature_names=feature_names))
# 想一想：規則有變嗎？為什麼？

---
## B・換上真的股票資料

那 6 家太乾淨了（一刀就分完），看不出真實的難處。這一段換成**真股票**——資料量大、很亂、也更誠實。**C 段全部用這份真資料。**

### B1・改用真的股票資料

Part A 那 6 家是手編的、乾淨到一眼看得出漲跌。現在換**真的**——用 Yahoo Finance 抓 8 檔台灣大型股（台積電、鴻海、聯發科…）三年的每日價量，算成 5 個「技術指標」，答案是「**明天**會漲還是跌」。

- 每一列 ＝ 某一檔股票某一天。
- 5 個特徵都是從價量算出來的：近 5 日／近 20 日漲跌幅、離 20 日均線多遠、近期波動大小、成交量放大幾倍。

**預期輸出：** 約 5648 筆（8 檔 × 三年交易日）；漲、跌大約各半。

> 💡 **第一次跑會上網抓，抓完存成 `data/prices_cache.csv`；之後（還有沒網路時）直接讀那個檔** —— 所以全班數字一致、離線也跑得動。

In [ ]:
import os
import pandas as pd

cache_path = "data/prices_cache.csv"

if os.path.exists(cache_path):
    # 已經有現成的資料就直接讀 —— 全班一致、沒網路也能跑
    data = pd.read_csv(cache_path, parse_dates=["date"])
    print("讀到現成資料：", cache_path)
else:
    # 第一次跑：從 Yahoo Finance 抓 8 檔真股票，算成特徵表再存起來
    import yfinance as yf

    tickers = ["2330.TW", "2317.TW", "2454.TW", "2308.TW",
               "2382.TW", "2412.TW", "2881.TW", "2882.TW"]
    name_map = {"2330.TW": "台積電", "2317.TW": "鴻海", "2454.TW": "聯發科",
                "2308.TW": "台達電", "2382.TW": "廣達", "2412.TW": "中華電",
                "2881.TW": "富邦金", "2882.TW": "國泰金"}

    # auto_adjust=True：把除權息、減資後的價格接成一條連續的線
    raw = yf.download(tickers, start="2022-01-01", end="2024-12-31",
                      progress=False, auto_adjust=True)

    parts = []
    for code in tickers:
        one = pd.DataFrame({
            "close": raw["Close"][code],
            "vol": raw["Volume"][code],
        }).dropna()
        # 5 個技術指標特徵（都是從價、量算出來的，不是財報數字）
        one["ret5"] = one["close"].pct_change(5)                    # 近 5 日漲跌幅
        one["ret20"] = one["close"].pct_change(20)                  # 近 20 日漲跌幅
        ma20 = one["close"].rolling(20).mean()                      # 20 日均線
        one["ma_bias"] = one["close"] / ma20 - 1                    # 現價離均線多遠
        one["vol20"] = one["close"].pct_change().rolling(20).std()  # 近 20 日波動大小
        one["volr"] = one["vol"] / one["vol"].rolling(20).mean()    # 成交量比平常放大幾倍
        # 答案：明天收盤比今天高 → 1（漲）、否則 → 0（跌）
        one["label"] = (one["close"].shift(-1) > one["close"]).astype(int)
        one = one.dropna()
        one["ticker"] = name_map[code]
        one["date"] = one.index
        parts.append(one[["date", "ticker", "ret5", "ret20", "ma_bias", "vol20", "volr", "label"]])

    data = pd.concat(parts).sort_values("date").reset_index(drop=True)
    for col in ["ret5", "ret20", "ma_bias", "vol20", "volr"]:
        data[col] = data[col].round(4)
    data.to_csv(cache_path, index=False)
    print("已下載並存檔：", cache_path)

# 今天用這 5 個特徵；每一列 = 某一檔股票某一天
feature_names = ["ret5", "ret20", "ma_bias", "vol20", "volr"]
X = data[feature_names].values
y = data["label"].values

print("總筆數：", len(data), "（8 檔股票 × 三年交易日）")
print("漲（1）跌（0）各幾筆：")
print(data["label"].value_counts())
print(data.head())

### B2・故意犯一個錯

拿全部資料訓練一台 KNN，然後**用同一批資料考它**，看準確率。

**預期輸出：** 大約 0.69。

> ⚠️ **看起來比擲銅板（0.5）好不少？先別高興。** 我們用同一批資料訓練、又用同一批資料考試——這是「拿念過的考卷考自己」。等一下 C 段改用**沒看過**的資料考，這個數字會掉下來。

In [ ]:
knn_all = KNeighborsClassifier(n_neighbors=5)
# TODO：故意「用全部資料訓練、又拿同一批考」——兩個空都填「全部那份特徵 X、答案 y」
knn_all.fit(____, ____)                             # 用全部資料訓練

print("用全部資料訓練，再回頭考同一批：", round(knn_all.score(X, y), 3))
# 💡 這個分數等一下會被推翻 —— 它是「拿念過的考卷考自己」

---
## C・課程重點：兩台一起壞給你看

老師剛剛講了 A 同學和 B 同學的故事：一個把答案位置背死（念過的滿分、換張考卷就垮），一個只記「選最長的」（兩邊都不好）。

接下來要做的事，一句話講完：**KNN 的 `k` 和決策樹的 `max_depth` 各自轉到極端，看它們怎麼變成 A 同學或 B 同學。**

兩個新名字：
- 模型**念過**的那份資料叫 **train**（樣本內）；**沒看過、留著當考卷**的那份叫 **test**（樣本外）。

> 🎭 **老師剛剛講了小明兩個同學的故事：**
> - **A 同學**把 100 題的答案位置整個背下來 → 考古題滿分、換張新考卷就垮。
> - **B 同學**只記「選項最長的通常是答案」→ 兩邊都只有六十幾分。
>
> A 同學那種「背死、換題就垮」＝ **過擬合（overfitting）**；B 同學那種「太粗糙、兩邊都不好」＝ **欠擬合（underfitting）**。
> **接下來就是看 KNN 和決策樹，會不會也變成 A 同學。**

### C1・切一次 train/test（兩台共用）

**這一刀只切一次，KNN 和決策樹都用同一份。** 但真股票是**時間序列**——不能隨機抽樣切，否則等於拿 3 月的資料去考 1 月的題目（偷看未來）。所以改成**用一個日期切開：這天以前的拿去念書、以後的留著當考卷**。

**預期輸出：** 分界日約 2024-06-04，train 約 4512 筆、test 約 1136 筆。

In [ ]:
# 真股票是時間序列：不能隨機切，那等於拿未來考過去（偷看答案）
# 正確做法：用一個日期切開，這天以前的念書、以後的當考卷
data = data.sort_values("date").reset_index(drop=True)

cut = int(len(data) * 0.8)           # 前 80% 拿去訓練
split_date = data["date"].iloc[cut]  # 用這一天當分界

# TODO：兩個空都填「分界那一天」的變數 —— 之前的當 train、之後的當 test
train = data[data["date"] < ____]
test = data[data["date"] >= ____]

X_train = train[feature_names].values
y_train = train["label"].values
X_test = test[feature_names].values
y_test = test["label"].values

print("分界日期：", split_date.date())
print("train（念書，較早的日子）：", len(X_train), "筆")
print("test （考試，較晚的日子）：", len(X_test), "筆")

### C2・先看單一設定的樣本內外差距

各建一台（KNN 用 k=5、決策樹用深度 3），分別考「念過的」和「沒看過的」。

**預期輸出：** KNN 樣本內約 0.69、樣本外約 0.53；決策樹兩邊都在 0.5 上下。**兩台的樣本內都比樣本外高。**

> ⚠️ **`fit` 只餵 `X_train`** —— 那 1136 筆考卷，訓練時完全不能給它看。

In [ ]:
knn5 = KNeighborsClassifier(n_neighbors=5)
knn5.fit(X_train, y_train)

tree3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree3.fit(X_train, y_train)

for name, model in [("KNN k=5     ", knn5), ("決策樹 深度3", tree3)]:
    # TODO：樣本內＝考「念過的 train」；樣本外＝考「沒看過的 test」
    in_score = model.score(____, ____)            # 樣本內：考念過的
    out_score = model.score(____, ____)           # 樣本外：考沒看過的
    print(name, "→ 樣本內", round(in_score, 3), " 樣本外", round(out_score, 3),
          " 差距", round(in_score - out_score, 3))

### C3・⭐ 把 KNN 的 k 從小掃到大

一次跑 8 個 k，每個都印樣本內、樣本外、以及兩者的差距。差距超過 0.1 就標記出來。

**預期輸出：8 行。盯著第一行看。**

> ⚠️ **`k=1` 的樣本內是 1.0（100% 全對），樣本外卻只有 0.51。** 停在這裡想三秒：這是好消息還是壞消息？

In [ ]:
ks = [1, 3, 5, 7, 9, 15, 25, 49]

print("k | 樣本內 | 樣本外 | 差距")
for k in ks:
    # TODO：這一圈要測的 k（就是迴圈變數）
    m = KNeighborsClassifier(n_neighbors=____)
    m.fit(X_train, y_train)
    in_score = m.score(X_train, y_train)
    out_score = m.score(X_test, y_test)
    gap = round(in_score - out_score, 3)
    if gap > 0.1:
        flag = "  ← 過擬合!"
    else:
        flag = ""
    print(k, "|", round(in_score, 3), "|", round(out_score, 3), "|", gap, flag)
# 想一想：k=1 樣本內 100%，是好消息還是壞消息？

### C4・⭐ 換一台模型、換一個旋鈕——看會不會發生同樣的事

同一份資料、同一個切法，這次換決策樹，掃它的 `max_depth`。

**預期輸出：6 行。**

> ⚠️ **`max_depth=None` 的樣本內又是 1.0。** 把這張表和上一張擺在一起看——它們的共同點是什麼？

In [ ]:
depths = [1, 2, 3, 5, 10, None]     # None ＝ 不限制深度，讓樹一直長下去

print("max_depth | 樣本內 | 樣本外 | 差距")
for d in depths:
    # TODO：這一圈要測的深度（就是迴圈變數）
    m = DecisionTreeClassifier(max_depth=____, random_state=42)
    m.fit(X_train, y_train)
    in_score = m.score(X_train, y_train)
    out_score = m.score(X_test, y_test)
    gap = round(in_score - out_score, 3)
    if gap > 0.1:
        flag = "  ← 過擬合!"
    else:
        flag = ""
    print(d, "|", round(in_score, 3), "|", round(out_score, 3), "|", gap, flag)
# 想一想：這張表和上一張的共同點是什麼？

### C5-1・自動挑出 KNN 該用的 k

規則升級一點：**挑樣本外最高的——但樣本內 100% 的那個是背起來的，直接跳過。**

**預期輸出：** k = 5，樣本外約 0.53。

> 💬 就算挑到最好的，樣本外也只比擲銅板好兩三個百分點——真股票的隔日漲跌，本來就很難猜。

In [ ]:
best_k = None
best_out = -1

for k in ks:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train, y_train)
    in_score = m.score(X_train, y_train)
    out_score = m.score(X_test, y_test)
    if in_score > 0.99:               # 💡 樣本內幾乎全對 = 把答案背死了，這種不能選
        continue
    if out_score > best_out:          # 剩下的裡面，挑樣本外最高的
        best_out = out_score
        best_k = k

print("KNN 該選的 k =", best_k, " 樣本外 =", round(best_out, 3))

### C5-2・同一招，換決策樹再做一次

**一模一樣的寫法，只把 `n_neighbors` 換成 `max_depth`。** 一樣要跳過樣本內 100% 的（就是 C4 裡 `max_depth=None` 那個背死的）。

**預期輸出：** max_depth = 1，樣本外約 0.51。

> 💬 甜蜜點落在最淺那一端；就算是它，樣本外也只有 5 成出頭。**找它的方法永遠一樣——但這份資料誠實地告訴你：光靠這幾個技術指標，隔日漲跌就是猜不動。這正是下一個 Lab 要加「新聞情緒特徵」的原因。**

In [ ]:
best_d = None
best_out_d = -1

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    in_score = m.score(X_train, y_train)
    out_score = m.score(X_test, y_test)
    if in_score > 0.99:               # 💡 同一條規則：樣本內背死的先跳過
        continue
    if out_score > best_out_d:
        best_out_d = out_score
        best_d = d

print("決策樹 該選的 max_depth =", best_d, " 樣本外 =", round(best_out_d, 3))

### C6・回頭看特徵重要性：這次不極端了

Part A 那 6 家太乾淨，重要性是極端的 1.0 / 0.0 / 0.0。換成這幾千筆真股票資料再看一次。

**預期輸出：** 5 個技術指標把重要性分掉了（近 20 日漲跌幅 `ret20` 最關鍵，約 0.30；沒有單一指標獨大）。**真實資料長這樣。**

> 💬 記住這張表的長相——**下一個 Lab 會在這裡多冒出一欄「新聞情緒分數」**，看它能不能擠進前面。

In [ ]:
d3 = DecisionTreeClassifier(max_depth=3, random_state=42)
d3.fit(X_train, y_train)

for i in range(len(feature_names)):
    print(feature_names[i], "重要性：", round(d3.feature_importances_[i], 3))
# 💡 資料一變亂，三個指標就都派上用場了 —— 不再是某一個獨大

### 📝 小作業 C

把 C1 的切分比例從前 `80%` 改成前 `70%`（`cut` 用 `0.7`），重新掃一次 k。**最好的 k 還是 5 嗎？**

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
cut2 = int(len(data) * 0.7)            # 改成前 70% 當訓練
split2 = data["date"].iloc[cut2]
train2 = data[data["date"] < split2]
test2 = data[data["date"] >= split2]
Xtr2 = train2[feature_names].values
ytr2 = train2["label"].values
Xte2 = test2[feature_names].values
yte2 = test2["label"].values

best = None
best_score = -1
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(Xtr2, ytr2)
    in_s = m.score(Xtr2, ytr2)
    out_s = m.score(Xte2, yte2)
    print("k =", k, " 樣本外", round(out_s, 3))
    if in_s > 0.99:
        continue
    if out_s > best_score:
        best_score = out_s
        best = k
print("改成前 70% 時最好的 k =", best)
```
**結論：從 5 變成 3 了。** 切法一改，甜蜜點就跑掉——**所以「k 要設 5」不是可以背起來的知識**，換一份資料、換一個切法就得重找。而且不管怎麼切，樣本外都還是貼著 5 成。

**能帶走的不是那個數字，是找它的方法：切 train/test → 掃旋鈕 → 跳過背死的、挑樣本外最高。**
</details>

In [ ]:
# 📝 小作業 C：你的答案（把切分比例改成前 70%，重新找最好的 k）

# TODO：把 cut 的比例改成 0.7（前 70% 當訓練）
cut2 = int(len(data) * ____)
split2 = data["date"].iloc[cut2]
train2 = data[data["date"] < split2]
test2 = data[data["date"] >= split2]
Xtr2 = train2[feature_names].values
ytr2 = train2["label"].values
Xte2 = test2[feature_names].values
yte2 = test2["label"].values

best = None
best_score = -1
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(Xtr2, ytr2)
    in_s = m.score(Xtr2, ytr2)
    out_s = m.score(Xte2, yte2)
    print("k =", k, " 樣本外", round(out_s, 3))
    if in_s > 0.99:
        continue
    if out_s > best_score:
        best_score = out_s
        best = k

print("改成前 70% 時最好的 k =", best)
# 想一想：最好的 k 還是 5 嗎？

### 📝 小作業 C-2（⭐ 進階選做）

拿 C5-1 選出的 `best_k` 和 C5-2 選出的 `best_d`，各建一台，**直接比誰的樣本外分數高**。

想一想：贏的那台，就是「比較好的模型」嗎？

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
a = KNeighborsClassifier(n_neighbors=best_k)
a.fit(X_train, y_train)
b = DecisionTreeClassifier(max_depth=best_d, random_state=42)
b.fit(X_train, y_train)

print("KNN   k =", best_k, " 樣本外 =", round(a.score(X_test, y_test), 3))
print("決策樹 max_depth =", best_d, " 樣本外 =", round(b.score(X_test, y_test), 3))
```
**結論：** KNN 0.527、決策樹 0.511，KNN 小贏。

**但先別下結論說「KNN 比較好」** —— 差距只有 0.016，而且兩台都只比擲銅板好一點點；這還是**一次切法**的結果（小作業 C 才剛示範過切法一換數字就跑）。真要比模型，得換很多種切法反覆比才算數。
決策樹雖然小輸，但它**印得出規則、說得出哪個指標重要**——在金融、法遵這種要跟人解釋的場合，這件事常常比 0.016 值錢。
</details>

In [ ]:
# 📝 小作業 C-2：你的答案（兩台各用自己選出的甜蜜點，比樣本外）

# TODO：用 C5-1 選出的 best_k、C5-2 選出的 best_d
a = KNeighborsClassifier(n_neighbors=____)
a.fit(X_train, y_train)
b = DecisionTreeClassifier(max_depth=____, random_state=42)
b.fit(X_train, y_train)

print("KNN   k =", best_k, " 樣本外 =", round(a.score(X_test, y_test), 3))
print("決策樹 max_depth =", best_d, " 樣本外 =", round(b.score(X_test, y_test), 3))
# 想一想：贏的那台，就一定是「比較好的模型」嗎？

---
## D・收尾

今天你親手做了**兩台**「用技術指標猜漲跌」的分類器，而且是在同一份真股票資料、同一個切法上做的：

```
真股票技術指標  →  用日期切 train/test  →  ┌─ KNN：掃 k
                                          └─ 決策樹：掃 max_depth（＋印規則、看哪個指標重要）
                                                 ↓
                                  兩台都是「樣本內衝到 100%、樣本外貼著 50%」
                                                 ↓
                                挑設定 ＝ 跳過背死的、選樣本外最高的
```

**⭐ 今天最該帶走的一句話：**
> 兩台完全不同的模型，用完全不同的旋鈕，卻用同一種方式壞掉。
> 所以「樣本內漂亮不代表真的好」不是某一台模型的毛病，**它是通則**。
> 你以後碰到任何模型、任何旋鈕，都先問這一句：**樣本外呢？**

**這條線你已經走過三段了：**
1. 課程 1 的 precision / recall —— 別用自己標的答案自我感覺良好。
2. 上一個 Lab 的時序分割 —— 別拿未來的資料考過去。
3. **今天** —— 別用念過的考卷考自己。

**還有一個真實世界的教訓：** 就算方法全做對，這兩台的樣本外也只有 5 成出頭——**光靠價量技術指標，隔日漲跌本來就很難猜**。這不是你做錯，是這題難。

**下一個 Lab 接什麼：** 今天的特徵只有價量算出來的技術指標。但有些影響漲跌的東西數字抓不到，比如「新聞講這家公司是利多還是利空」。後面會用 LLM 把新聞文字算成一個**情緒分數**，當作第 6 個特徵塞進今天這兩台，比較「加之前 vs 加之後」，並在 C6 那張重要性表上看它排第幾。

> 🚫 **再說一次：** 今天的「猜漲跌」純教學示範，**不是投資建議**。分類模型在金融上是輔助參考，最終決策要靠人。**你學到的真本事是「會建模型 + 會誠實驗證模型」，不是「跟模型下單」。**

---
## 🛟 Backup：卡住的時候

| 狀況 | 怎麼辦 |
|---|---|
| `No module named 'sklearn'` 或 `'yfinance'` | 回去跑第一格的 `!pip install`。**套件名是 `scikit-learn`，import 名是 `sklearn`**，別打錯。 |
| 抓不到網路 / yfinance 壞掉 | 只要 `data/prices_cache.csv` 在，就**完全不需要網路**——第一格會自動改讀它。這個檔 repo 裡已經附了。 |
| 我跑出的數字跟老師不一樣 | 多半是 `data/prices_cache.csv` 被刪了、程式重新上網抓（Yahoo 會隨新股利微調過去價格）。用 repo 附的那份快取就會一致；想重抓就把它刪掉再跑。決策樹的數字還要記得帶 `random_state=42`。 |
| `max_depth=None` 怎麼寫 | `None` 是 Python 的「沒有值」，代表不限制深度。清單裡直接放 `None`，**不要寫成字串 `"None"`**。 |
| `export_text` 印出 `feature_0` | 忘了傳 `feature_names=feature_names`。 |
| 重要性某個是 0.0 | 不是壞掉——代表那個指標沒被拿來分岔。 |
| 真的想拿它去預測股票 | **別。** 樣本外才 5 成，跟擲銅板差不多。今天學的是「怎麼誠實驗證模型」，不是「這台能拿去下單」。 |
| 想在真實資料上把 KNN 做對 | KNN 靠距離，怕某個特徵數字特別大。正式做要先縮放：`from sklearn.preprocessing import StandardScaler`，`scaler = StandardScaler().fit(X_train)`，再 `X_train = scaler.transform(X_train)`、`X_test = scaler.transform(X_test)`（**scaler 只能 fit train**）。 |

**下面這格是「最小可跑核心」——任何環境想確認兩台通不通，跑它就對了。**

In [ ]:
# 🛟 最小可跑核心：兩行都印出 [0 1] 就代表 sklearn 的 KNN 和決策樹都正常
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

check_knn = KNeighborsClassifier(n_neighbors=3)
check_knn.fit([[0, 0], [0, 1], [10, 10], [10, 11]], [0, 0, 1, 1])
print(check_knn.predict([[0, 0.5], [10, 10.5]]))

check_tree = DecisionTreeClassifier(max_depth=1, random_state=42)
check_tree.fit([[0], [1], [10], [11]], [0, 0, 1, 1])
print(check_tree.predict([[0.5], [10.5]]))